# EEG Data Exploration
**Project**: Robustness of EEG Feature Representations Under Cross-Dataset Distribution Shift  
**Purpose**: Visual sanity checks on raw data, preprocessing effects, and feature distributions.  
This notebook does not train any models — it exists purely to show what the data looks like at each stage of the pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import mne
from scipy.signal import welch
from moabb.datasets import PhysionetMI, BNCI2014_001

mne.set_log_level('WARNING')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
SEED = 42
np.random.seed(SEED)
print('All imports OK')

---
## 1. Load One Subject from Each Dataset
We load subject 1 from each dataset for visual inspection. This is the same loading pathway used in `preprocess.py`.

In [ ]:
# Load PhysioNet subject 1
physionet = PhysionetMI()
sessions_phys = physionet.get_data(subjects=[1])
raw_phys = list(list(sessions_phys[1].values())[0].values())[0]

print('=== PhysioNet Subject 1 ===')
print(f'Sampling rate: {raw_phys.info["sfreq"]} Hz')
print(f'Channels: {len(raw_phys.ch_names)}')
print(f'Duration: {raw_phys.times[-1]:.1f} s')
print(f'Labels: {set(raw_phys.annotations.description)}')

In [ ]:
# Load BCI2a subject 1
bci2a = BNCI2014_001()
sessions_bci = bci2a.get_data(subjects=[1])
raw_bci = list(list(sessions_bci[1].values())[0].values())[0]

print('=== BCI2a Subject 1 ===')
print(f'Sampling rate: {raw_bci.info["sfreq"]} Hz')
print(f'Channels: {len(raw_bci.ch_names)}')
print(f'Duration: {raw_bci.times[-1]:.1f} s')
print(f'Labels: {set(raw_bci.annotations.description)}')

---
## 2. Raw EEG Signal — What Brain Data Looks Like
Plot 5 seconds of raw EEG from the central channels (C3, Cz, C4). These are the channels most relevant to motor imagery — located directly over the motor cortex.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
plot_channels = ['C3', 'Cz', 'C4']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for ax, (raw, title) in zip(axes, [(raw_phys, 'PhysioNet (160 Hz, raw)'),
                                     (raw_bci,  'BCI2a (250 Hz, raw)')]):
    sfreq = raw.info['sfreq']
    n_samples = int(5 * sfreq)  # 5 seconds
    times = np.arange(n_samples) / sfreq

    for ch_name, color in zip(plot_channels, colors):
        ch_idx = raw.ch_names.index(ch_name)
        signal = raw.get_data(picks=[ch_idx])[0, :n_samples]
        ax.plot(times, signal * 1e6, label=ch_name, color=color, linewidth=0.8, alpha=0.9)

    ax.set_title(title)
    ax.set_ylabel('Amplitude (μV)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Raw EEG Signal — Central Motor Channels', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/explore_01_raw_signal.png', dpi=120, bbox_inches='tight')
plt.show()
print('Note the ~10x amplitude difference between PhysioNet and BCI2a.')
print('This is the 9x distribution shift identified in our diagnostic analysis.')

---
## 3. Effect of Bandpass Filtering (4–40 Hz)
Show the same signal before and after the 4–40 Hz bandpass filter. The filter removes slow drifts and high-frequency muscle noise, leaving only the motor imagery-relevant frequency range.

In [ ]:
BCI2A_CHANNELS = [
    'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
    'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
    'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
    'P1', 'Pz', 'P2', 'POz'
]

# Prepare PhysioNet: pick channels and resample
raw_before = raw_phys.copy().pick_channels(BCI2A_CHANNELS).resample(250)
raw_after  = raw_before.copy().filter(4.0, 40.0, verbose=False)

ch_idx = raw_before.ch_names.index('C3')
n_samples = int(3 * 250)  # 3 seconds at 250 Hz
times = np.arange(n_samples) / 250

sig_before = raw_before.get_data(picks=[ch_idx])[0, :n_samples] * 1e6
sig_after  = raw_after.get_data(picks=[ch_idx])[0, :n_samples] * 1e6

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].plot(times, sig_before, color='#e74c3c', linewidth=0.8)
axes[0].set_title('Channel C3 — Before Bandpass Filter')
axes[0].set_ylabel('Amplitude (μV)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(times, sig_after, color='#2ecc71', linewidth=0.8)
axes[1].set_title('Channel C3 — After 4–40 Hz Bandpass Filter')
axes[1].set_ylabel('Amplitude (μV)')
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of Bandpass Filtering on EEG Signal (PhysioNet Subject 1, C3)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/explore_02_filtering.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Power Spectral Density — Frequency Content
Show the PSD of channel C3 for both datasets. The mu rhythm (8–12 Hz) and beta rhythm (13–30 Hz) should be visible as peaks or plateaus. These are the frequencies that suppress during motor imagery.

In [ ]:
raw_bci_filt = raw_bci.copy().pick_channels(BCI2A_CHANNELS).filter(4.0, 40.0, verbose=False)

ch_idx_phys = raw_after.ch_names.index('C3')
ch_idx_bci  = raw_bci_filt.ch_names.index('C3')

sig_phys = raw_after.get_data(picks=[ch_idx_phys])[0]
sig_bci  = raw_bci_filt.get_data(picks=[ch_idx_bci])[0]

freqs_p, psd_p = welch(sig_phys, fs=250, nperseg=512)
freqs_b, psd_b = welch(sig_bci,  fs=250, nperseg=512)

mask_p = (freqs_p >= 1) & (freqs_p <= 45)
mask_b = (freqs_b >= 1) & (freqs_b <= 45)

fig, ax = plt.subplots(figsize=(10, 5))

ax.semilogy(freqs_p[mask_p], psd_p[mask_p], color='#e74c3c', label='PhysioNet S1', linewidth=1.5)
ax.semilogy(freqs_b[mask_b], psd_b[mask_b], color='#3498db', label='BCI2a S1',    linewidth=1.5)

# Shade frequency bands
band_colors = {'delta\n(0.5–4)': (0.5, 4, '#f39c12'),
               'theta\n(4–8)':   (4,   8, '#9b59b6'),
               'alpha/mu\n(8–13)': (8, 13, '#2ecc71'),
               'beta\n(13–30)':  (13, 30, '#e74c3c')}

for label, (fmin, fmax, color) in band_colors.items():
    ax.axvspan(fmin, fmax, alpha=0.08, color=color)
    ax.text((fmin + fmax) / 2, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 1e-20,
            label, ha='center', fontsize=8, alpha=0.7)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density (log scale)')
ax.set_title('PSD Comparison — Channel C3, Both Datasets (filtered 4–40 Hz)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(1, 45)

plt.tight_layout()
plt.savefig('../results/explore_03_psd.png', dpi=120, bbox_inches='tight')
plt.show()
print('Note: PhysioNet PSD is much higher than BCI2a — this is the amplitude distribution shift.')
print('Both datasets show similar frequency shape — this is why frequency features may be more robust.')

---
## 5. Single Trial — Left vs Right Hand Imagery
Extract one left-hand and one right-hand trial from the preprocessed data and plot them side by side. This shows what the model is trying to distinguish.

In [ ]:
X_bci = np.load('../results/bci2a_X.npy')
y_bci = np.load('../results/bci2a_y.npy')

# Get one left and one right trial
left_idx  = np.where(y_bci == 0)[0][0]
right_idx = np.where(y_bci == 1)[0][0]

ch_names = BCI2A_CHANNELS
c3_idx   = ch_names.index('C3')
cz_idx   = ch_names.index('Cz')
c4_idx   = ch_names.index('C4')
times    = np.linspace(0.5, 2.5, X_bci.shape[2])

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, idx, label, color in [
    (axes[0], left_idx,  'Left Hand Imagery',  '#3498db'),
    (axes[1], right_idx, 'Right Hand Imagery', '#e74c3c')
]:
    trial = X_bci[idx] * 1e6  # convert to μV
    for ch_idx, ch_name, alpha in [(c3_idx, 'C3', 0.9), (cz_idx, 'Cz', 0.6), (c4_idx, 'C4', 0.9)]:
        ax.plot(times, trial[ch_idx], label=ch_name, alpha=alpha, linewidth=1)
    ax.set_title(label, color=color, fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Epoch start')
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[0].set_ylabel('Amplitude (μV)')
plt.suptitle('Single Trial Comparison — BCI2a Preprocessed Data (0.5–2.5s post-cue)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/explore_04_single_trial.png', dpi=120, bbox_inches='tight')
plt.show()
print('The difference between left and right trials is subtle at the single-trial level.')
print('This is why classification accuracy is moderate — motor imagery is a weak signal.')

---
## 6. Feature Distribution Comparison
Compare how the three feature sets distribute across both datasets. If distributions are similar, cross-dataset transfer is easier. If they differ widely, transfer is harder.

In [ ]:
phys_A = np.load('../results/phys_A.npy')
bci_A  = np.load('../results/bci_A.npy')
phys_B = np.load('../results/phys_B.npy')
bci_B  = np.load('../results/bci_B.npy')
phys_C = np.load('../results/phys_C.npy')
bci_C  = np.load('../results/bci_C.npy')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Feature A — variance (column 22 onwards are variances)
axes[0].hist(phys_A[:, 22], bins=50, alpha=0.6, color='#e74c3c', label='PhysioNet', density=True)
axes[0].hist(bci_A[:, 22],  bins=50, alpha=0.6, color='#3498db', label='BCI2a',     density=True)
axes[0].set_title('Feature A — Variance\n(channel C3)')
axes[0].set_xlabel('Variance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature B — alpha band power (index 2 = alpha band for channel 0)
axes[1].hist(phys_B[:, 2], bins=50, alpha=0.6, color='#e74c3c', label='PhysioNet', density=True)
axes[1].hist(bci_B[:, 2],  bins=50, alpha=0.6, color='#3498db', label='BCI2a',     density=True)
axes[1].set_title('Feature B — Log Alpha Power\n(channel Fz)')
axes[1].set_xlabel('Log Band Power')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Feature C — FFT bin 10 (mid-frequency)
axes[2].hist(phys_C[:, 10], bins=50, alpha=0.6, color='#e74c3c', label='PhysioNet', density=True)
axes[2].hist(bci_C[:, 10],  bins=50, alpha=0.6, color='#3498db', label='BCI2a',     density=True)
axes[2].set_title('Feature C — FFT Magnitude\n(bin 10, ~14 Hz)')
axes[2].set_xlabel('FFT Magnitude')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Feature Distribution Comparison — PhysioNet vs BCI2a', fontsize=13)
plt.tight_layout()
plt.savefig('../results/explore_05_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

print('Observations:')
print(f'  Feature A variance: PhysioNet range {phys_A[:,22].max():.2e}, BCI2a range {bci_A[:,22].max():.2e}')
print(f'  Feature B log alpha: PhysioNet mean {phys_B[:,2].mean():.2f}, BCI2a mean {bci_B[:,2].mean():.2f}')
print(f'  Feature C FFT bin: PhysioNet mean {phys_C[:,10].mean():.4f}, BCI2a mean {bci_C[:,10].mean():.4f}')
print()
print('Feature A shows large distribution mismatch — consistent with weak transfer robustness.')
print('Feature B shows near-identical distributions — log transform absorbed much of the amplitude shift.')
print('Feature C shows moderate mismatch — raw FFT magnitude still reflects some amplitude difference, yet retained the strongest overall transfer performance in the final results.')

---
## 7. Summary

This notebook has shown:

1. **Raw signal** — EEG looks like oscillating noisy signals in the ±100–1000 μV range. PhysioNet signals are ~10x larger than BCI2a due to hardware differences.

2. **Filtering effect** — The 4–40 Hz bandpass removes slow drifts and high-frequency noise, leaving a cleaner signal centred around zero.

3. **PSD** — Both datasets show similar frequency shape (mu/beta peaks visible), but different absolute power levels. This is the distribution shift.

4. **Single trials** — Left vs right hand imagery looks nearly identical at the single-trial level. Motor imagery is a subtle signal.

5. **Feature distributions** — Feature A (variance) shows large cross-dataset mismatch. Feature B (log band power) shows near-identical distributions. Feature C (FFT) shows moderate mismatch. These plots help explain the transfer behavior, but they do not fully determine the final ranking by themselves.

These visualisations support two complementary conclusions: log-transformed band power partially absorbs the amplitude distribution shift between recording systems, while FFT still achieved the smallest overall normalised generalisation gap in the final experiments.